In [5]:
import pandas as pd

ELEC_CSV  = '/content/API_EG.ELC.ACCS.ZS_DS2_en_csv_v2_127016.csv'
EMPL_CSV  = '/content/API_SL.IND.EMPL.ZS_DS2_en_csv_v2_1355.csv'
MANUF_CSV = '/content/API_NV.IND.MANF.ZS_DS2_en_csv_v2_250129.csv'
GDP_CSV   = '/content/API_NY.GDP.PCAP.KD_DS2_en_csv_v2_260337.csv'
META_CSV  = '/content/Metadata_Country_API_EG.ELC.ACCS.ZS_DS2_en_csv_v2_127016.csv'
YEARS = [str(y) for y in range(2000, 2023)]

def load(path, varname):
  df = pd.read_csv(path, skiprows=4)
  df = df.dropna(subset=['Country Name'])
  keep = ['Country Name', 'Country Code'] + [y for y in YEARS if y in df.columns]
  df = df[keep]
  df = df.melt(id_vars=['Country Name', 'Country Code'], var_name='year', value_name=varname)
  df['year'] = df['year'].astype(int)
  return df

elec  = load(ELEC_CSV,  'elec_access')
empl  = load(EMPL_CSV,  'ind_employment')
manuf = load(MANUF_CSV, 'manuf_value_added')
gdp   = load(GDP_CSV,   'gdp_per_capita')

panel = elec.merge(empl,  on=['Country Name', 'Country Code', 'year'], how='outer')
panel = panel.merge(manuf, on=['Country Name', 'Country Code', 'year'], how='outer')
panel = panel.merge(gdp,   on=['Country Name', 'Country Code', 'year'], how='outer')

meta = pd.read_csv(META_CSV)[['Country Code', 'Region', 'IncomeGroup']]
panel = panel.merge(meta, on='Country Code', how='left')

developing_countries = panel[panel['IncomeGroup'].isin(['Low income','Lower middle income','Upper middle income'
])].copy()

developing_countries = developing_countries.dropna(
    subset=['elec_access', 'ind_employment', 'manuf_value_added', 'gdp_per_capita'],
    how='all'
)

developing_countries = developing_countries.rename(columns={
    'Country Name': 'country',
    'Country Code': 'country_code'
})
developing_countries = developing_countries.sort_values(['country', 'year']).reset_index(drop=True)

developing_countries.to_csv('/content/QSS20_panel_clean.csv', index=False)
print(f"Done. Shape: {developing_countries.shape}")
print(f"Countries: {developing_countries['country'].nunique()}")
print(f"Years: {developing_countries['year'].min()}–{developing_countries['year'].max()}")

Done. Shape: (2959, 9)
Countries: 129
Years: 2000–2022
